# Hull Tactical — Compact ML Project v3

This version keeps the project compact but adds the important methodological part:

```text
Each model is not used with random parameters.
Several parameter values are tested.
The best parameters are selected using the validation Model Score.
```

The goal is not only to predict returns.  
The goal is to create an allocation strategy that performs well and respects the volatility constraint.

Important allocation interpretation:

```text
0 = no S&P 500 exposure
1 = normal 100% exposure
2 = 200% leveraged exposure
```

The allocation can also be continuous, for example `0.4`, `0.9`, `1.3`, or `1.8`.

## 1. Import libraries and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

TARGET = "market_forward_excess_returns"

train = pd.read_csv("train.csv").sort_values("date_id").reset_index(drop=True)
test = pd.read_csv("test.csv").sort_values("date_id").reset_index(drop=True)

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

## 2. Data analysis: why these models?

The target is the future daily excess return of the S&P 500.

### Daily excess return

```text
daily_excess_return = market_return - baseline_return
```

In this dataset, the target column is already provided:

```text
market_forward_excess_returns
```

### Why this dataset is difficult

From the graphs below, we can see that:

1. Daily returns are very noisy.
2. Most values are close to zero.
3. There are outliers.
4. Volatility changes over time.

Because of this, the chosen models must be able to work with noisy tabular data.

### Why these algorithms?

- **Ridge Regression**: simple regularized baseline. Useful when many features are weak and noisy.
- **Hist Gradient Boosted Tree**: captures non-linear relationships and interactions between financial features.
- **Random Forest**: robust tree ensemble that reduces variance by averaging many trees.
- **Extra Trees**: similar to Random Forest but more randomized, which can help reduce overfitting.
- **Ensemble**: combines models because no single model is expected to capture all market behavior.

In [ ]:
y = train[TARGET].astype(float)

plt.figure(figsize=(10, 3))
plt.plot(train["date_id"], y)
plt.title("Target over time")
plt.xlabel("date_id")
plt.ylabel("daily excess return")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.hist(y, bins=70)
plt.title("Target distribution")
plt.xlabel("daily excess return")
plt.ylabel("number of days")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(train["date_id"], y.rolling(21).std())
plt.title("21-day rolling volatility")
plt.xlabel("date_id")
plt.ylabel("21-day standard deviation")
plt.tight_layout()
plt.show()

print(y.describe())

## 3. Compact data preparation

The dataset contains many financial indicators. Some early values are missing because the dataset covers many decades.

### Missing percentage formula

```text
missing_% = number of missing values / number of rows × 100
```

### Cleaning method

1. Recreate lagged features in train to match test.
2. Keep numeric columns available in both train and test.
3. Drop features where `missing_% > 60%`.
4. Add missing indicator columns.
5. Use forward fill.
6. Fill remaining beginning missing values with the training median.

### Forward fill idea

```text
if x_t is missing, use the most recent previous value
```

### Median fill idea

If no previous value exists yet, usually at the beginning of the dataset:

```text
fill with the training median
```

In [ ]:
# Create lagged columns in train because test has these columns.
train["lagged_forward_returns"] = train["forward_returns"].shift(1)
train["lagged_risk_free_rate"] = train["risk_free_rate"].shift(1)
train["lagged_market_forward_excess_returns"] = train[TARGET].shift(1)

# Keep only usable numeric features shared by train and test.
leakage = ["forward_returns", "risk_free_rate", TARGET, "is_scored", "date_id"]
all_features = [c for c in train.columns
                if c in test.columns
                and pd.api.types.is_numeric_dtype(train[c])
                and c not in leakage]

# Drop features with more than 60% missing values.
missing_rate = train[all_features].isna().mean()
features = missing_rate[missing_rate <= 0.60].index.tolist()
dropped_features = missing_rate[missing_rate > 0.60].index.tolist()

print("Features before missing filter:", len(all_features))
print("Dropped features with >60% missing:", len(dropped_features))
print("Features kept:", len(features))

X = train[features].copy()
X_test = test[features].copy()

# Missing indicators are added before filling.
for c in features:
    if X[c].isna().any():
        X[c + "_missing"] = X[c].isna().astype(int)
        X_test[c + "_missing"] = X_test[c].isna().astype(int)

# Forward fill, then fill remaining beginning missing values with training median.
train_median = X.median()
X = X.ffill().fillna(train_median)
X_test = X_test.ffill().fillna(train_median)

print("Feature count after missing indicators:", X.shape[1])

## 4. Feature engineering

New features are added because individual anonymized columns may be weak alone, but their combined behavior can be useful.

### Added features

```text
row_mean = average of selected features in one row
row_std = standard deviation of selected features in one row
row_missing_count = number of originally missing values in one row
```

Group means are also added:

```text
M_group_mean, E_group_mean, I_group_mean, P_group_mean, V_group_mean, S_group_mean, D_group_mean, MOM_group_mean
```

Date-cycle features are added because market behavior can have short-term, monthly, and yearly patterns:

```text
date_sin_period = sin(2π × date_id / period)
date_cos_period = cos(2π × date_id / period)
```

In [ ]:
def add_features(X, original_df, base_features):
    X = X.copy()

    X["row_mean"] = X[base_features].mean(axis=1)
    X["row_std"] = X[base_features].std(axis=1)

    missing_cols = [c for c in X.columns if c.endswith("_missing")]
    X["row_missing_count"] = X[missing_cols].sum(axis=1) if missing_cols else 0

    groups = {
        "MOM": [c for c in base_features if c.startswith("MOM")],
        "M": [c for c in base_features if c.startswith("M") and not c.startswith("MOM")],
        "E": [c for c in base_features if c.startswith("E")],
        "I": [c for c in base_features if c.startswith("I")],
        "P": [c for c in base_features if c.startswith("P")],
        "V": [c for c in base_features if c.startswith("V")],
        "S": [c for c in base_features if c.startswith("S")],
        "D": [c for c in base_features if c.startswith("D")],
        "lagged": [c for c in base_features if c.startswith("lagged_")]
    }

    for group_name, cols in groups.items():
        if len(cols) > 0:
            X[group_name + "_group_mean"] = X[cols].mean(axis=1)

    date_id = original_df["date_id"].astype(float)
    for period in [5, 21, 252]:
        X[f"date_sin_{period}"] = np.sin(2 * np.pi * date_id / period)
        X[f"date_cos_{period}"] = np.cos(2 * np.pi * date_id / period)

    return X

X = add_features(X, train, features)
X_test = add_features(X_test, test, features)

print("Final feature count after engineering:", X.shape[1])
X.head()

## 5. Missing values graph

The y-axis is the feature name.  
The x-axis is the percentage of missing values before filling.

Only features that were kept after the 60% filter are shown.

In [ ]:
missing = train[features].isna().mean().sort_values(ascending=False).head(20) * 100

plt.figure(figsize=(8, 5))
plt.barh(missing.index[::-1], missing.values[::-1])
plt.title("Top missing features kept in the model")
plt.xlabel("missing values (%)")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

## 6. Train/validation split

Even though a test set exists, it does not contain the target values.  
So we cannot use it to calculate RMSE, volatility, Sharpe, or Model Score.

Therefore, the training data is split chronologically:

```text
oldest 75% of train.csv = training set
newest 25% of train.csv = validation set
```

This simulates real life:

```text
train on the past → validate on the future
```

After the best model is selected, it is retrained on 100% of `train.csv` and then applied to `test.csv`.

In [ ]:
split = int(len(train) * 0.75)

X_train, X_val = X.iloc[:split], X.iloc[split:]
y_train, y_val = y.iloc[:split], y.iloc[split:]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

## 7. Evaluation formulas

The model predicts a daily excess return:

```text
prediction_t = predicted excess return for day t
```

The prediction is transformed into an allocation:

```text
signal_t = tanh(prediction_t / std(predictions))
```

```text
allocation_t = clip(1 + 0.65 × signal_t, 0, 2)
```

The allocation is continuous between 0 and 2.

Strategy daily return:

```text
strategy_return_t = allocation_t × actual_market_excess_return_t
```

Annual return:

```text
annual_return = mean(daily_returns) × 252
```

Annual volatility:

```text
annual_volatility = std(daily_returns) × sqrt(252)
```

Volatility ratio:

```text
volatility_ratio = strategy_annual_volatility / market_annual_volatility
```

The required constraint is:

```text
volatility_ratio ≤ 1.20
```

Sharpe ratio:

```text
Sharpe = strategy_annual_return / strategy_annual_volatility
```

Model Score:

```text
Model Score = Sharpe
              - 4 × max(0, volatility_ratio - 1.20)
              - 2 × max(0, market_return - strategy_return)
```

The best model is selected by **highest Model Score**, not only by lowest RMSE.

In [ ]:
def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

def allocation(pred):
    signal = np.tanh(pred / (np.std(pred) + 1e-12))
    return np.clip(1 + 0.65 * signal, 0, 2)

def metrics(y_true, alloc):
    strategy_daily = alloc * y_true
    market_daily = y_true

    strategy_return = strategy_daily.mean() * 252
    market_return = market_daily.mean() * 252

    strategy_vol = strategy_daily.std() * np.sqrt(252)
    market_vol = market_daily.std() * np.sqrt(252)

    vol_ratio = strategy_vol / (market_vol + 1e-12)
    sharpe = strategy_return / (strategy_vol + 1e-12)

    penalty = max(0, vol_ratio - 1.20) * 4 + max(0, market_return - strategy_return) * 2
    score = sharpe - penalty

    return strategy_return, market_return, strategy_vol, market_vol, vol_ratio, sharpe, score

def cumulative(r):
    return (1 + pd.Series(r)).cumprod() - 1

print("Evaluation functions ready.")

## 8. Hyperparameter tuning

Each model has parameters that must be chosen.

Examples:

```text
Ridge: alpha
Random Forest: max_depth, min_samples_leaf, max_features
Extra Trees: max_depth, min_samples_leaf, max_features
Hist Gradient Boosted Tree: learning_rate, max_leaf_nodes, min_samples_leaf
```

A small set of possible values is tested.  
For each configuration, the model is trained on the training period and evaluated on the validation period.

The best configuration is selected using the highest **Model Score**.

In [ ]:
candidates = {
    "Ridge": [
        {"alpha": 1},
        {"alpha": 10},
        {"alpha": 30},
        {"alpha": 100}
    ],

    "Hist Gradient Boosted Tree": [
        {"learning_rate": 0.02, "max_leaf_nodes": 7, "min_samples_leaf": 50},
        {"learning_rate": 0.03, "max_leaf_nodes": 15, "min_samples_leaf": 35},
        {"learning_rate": 0.05, "max_leaf_nodes": 15, "min_samples_leaf": 35},
        {"learning_rate": 0.03, "max_leaf_nodes": 31, "min_samples_leaf": 50}
    ],

    "Random Forest": [
        {"max_depth": 4, "min_samples_leaf": 50, "max_features": 0.50},
        {"max_depth": 6, "min_samples_leaf": 30, "max_features": 0.70},
        {"max_depth": 8, "min_samples_leaf": 50, "max_features": 0.70}
    ],

    "Extra Trees": [
        {"max_depth": 4, "min_samples_leaf": 50, "max_features": 0.50},
        {"max_depth": 6, "min_samples_leaf": 30, "max_features": 0.70},
        {"max_depth": 8, "min_samples_leaf": 25, "max_features": 0.70}
    ]
}

def build_model(name, params):
    if name == "Ridge":
        return make_pipeline(RobustScaler(), Ridge(alpha=params["alpha"]))

    if name == "Hist Gradient Boosted Tree":
        return HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=params["learning_rate"],
            max_leaf_nodes=params["max_leaf_nodes"],
            min_samples_leaf=params["min_samples_leaf"],
            l2_regularization=0.25,
            random_state=42
        )

    if name == "Random Forest":
        return RandomForestRegressor(
            n_estimators=250,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )

    if name == "Extra Trees":
        return ExtraTreesRegressor(
            n_estimators=300,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )

tuning_rows = []
best_models = {}
best_params = {}

for name, configs in candidates.items():
    best_score = -999

    for params in configs:
        model = build_model(name, params)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        val_alloc = allocation(val_pred)

        strategy_return, market_return, strategy_vol, market_vol, vol_ratio, sharpe, score = metrics(y_val, val_alloc)

        tuning_rows.append({
            "Model": name,
            "Parameters": str(params),
            "Train RMSE": rmse(y_train, train_pred),
            "Validation RMSE": rmse(y_val, val_pred),
            "Overfit ratio": rmse(y_val, val_pred) / (rmse(y_train, train_pred) + 1e-12),
            "Strategy return %": strategy_return * 100,
            "Vol ratio %": vol_ratio * 100,
            "Vol constraint": "PASS" if vol_ratio <= 1.20 else "FAIL",
            "Sharpe": sharpe,
            "Model Score": score
        })

        if score > best_score:
            best_score = score
            best_models[name] = build_model(name, params)
            best_params[name] = params

tuning_results = pd.DataFrame(tuning_rows).sort_values(["Model", "Model Score"], ascending=[True, False])
tuning_results.to_csv("tuning_results.csv", index=False)

best_parameter_table = tuning_results.loc[tuning_results.groupby("Model")["Model Score"].idxmax()]
best_parameter_table = best_parameter_table.sort_values("Model Score", ascending=False)

best_parameter_table

## 9. Best parameters selected

This table shows which parameter configuration was chosen for each model.

Important: the best parameters are chosen using **validation Model Score**, because the project goal is investment performance under a risk constraint.

In [ ]:
best_parameter_table[["Model", "Parameters", "Validation RMSE", "Vol ratio %", "Vol constraint", "Sharpe", "Model Score"]]

## 10. Train the best version of each model

Now each model family is trained using its selected best parameters.

In [ ]:
results = []
preds = {}
allocs = {}
fitted_models = {}

for name, model_template in best_models.items():
    model = clone(model_template)
    model.fit(X_train, y_train)
    fitted_models[name] = model

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    preds[name] = val_pred
    allocs[name] = allocation(val_pred)

    strategy_return, market_return, strategy_vol, market_vol, vol_ratio, sharpe, score = metrics(y_val, allocs[name])

    results.append({
        "Model": name,
        "Best parameters": str(best_params[name]),
        "Train RMSE": rmse(y_train, train_pred),
        "Validation RMSE": rmse(y_val, val_pred),
        "Overfit ratio": rmse(y_val, val_pred) / (rmse(y_train, train_pred) + 1e-12),
        "Strategy return %": strategy_return * 100,
        "Market return %": market_return * 100,
        "Vol ratio %": vol_ratio * 100,
        "Vol constraint": "PASS" if vol_ratio <= 1.20 else "FAIL",
        "Sharpe": sharpe,
        "Model Score": score
    })

results = pd.DataFrame(results).sort_values("Model Score", ascending=False)
results.to_csv("best_model_results.csv", index=False)
results

## 11. Model comparison graphs

### Volatility graph

- x-axis: model name
- y-axis: volatility ratio in percent
- dashed line: maximum allowed value, 120%

### Model Score graph

- x-axis: Model Score
- y-axis: model name
- higher score means better risk-adjusted performance

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results["Model"], results["Vol ratio %"])
plt.axhline(120, linestyle="--", label="120% limit")
plt.title("Volatility ratio by tuned model")
plt.ylabel("volatility ratio (%)")
plt.xlabel("model")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
score_sorted = results.sort_values("Model Score")
plt.barh(score_sorted["Model"], score_sorted["Model Score"])
plt.title("Model Score comparison")
plt.xlabel("Model Score (higher is better)")
plt.ylabel("model")
plt.tight_layout()
plt.show()

## 12. Prediction diagnostic and cumulative return

The most important graph for the project is the **cumulative return graph**, because the final objective is investment performance.

The prediction-vs-actual graph is kept only as a diagnostic:

- x-axis: validation day
- y-axis: daily excess return
- actual daily returns are noisy,
- predicted daily returns are smoother,
- so this graph is not the main result.

The main result is the cumulative return and the volatility constraint.

In [ ]:
best_individual_name = results.iloc[0]["Model"]

plot_days = min(250, len(y_val))

plt.figure(figsize=(10, 3))
plt.plot(y_val.values[:plot_days], label="Actual daily excess return")
plt.plot(preds[best_individual_name][:plot_days], label="Predicted daily excess return")
plt.title("Prediction diagnostic — " + best_individual_name)
plt.xlabel("validation day")
plt.ylabel("daily excess return")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(cumulative(y_val.values), label="Market")

for name in allocs:
    plt.plot(cumulative(allocs[name] * y_val.values), label=name)

plt.title("Cumulative return comparison")
plt.xlabel("validation day")
plt.ylabel("cumulative return")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Ensemble model

The ensemble combines the tuned individual model predictions.

```text
ensemble_prediction =
w1 × model1_prediction
+ w2 × model2_prediction
+ ...
```

The notebook searches for the weights that give the highest validation Model Score.

In [ ]:
model_names = list(preds.keys())
P = np.column_stack([preds[name] for name in model_names])

best_score = -999
best_w = None
best_pred = None
best_alloc = None

rng = np.random.default_rng(42)

for i in range(3000):
    w = rng.dirichlet(np.ones(len(model_names)))
    p = P @ w
    a = allocation(p)
    score = metrics(y_val, a)[-1]

    if score > best_score:
        best_score = score
        best_w = w
        best_pred = p
        best_alloc = a

weights = pd.DataFrame({"Model": model_names, "Weight": best_w}).sort_values("Weight", ascending=False)
weights.to_csv("ensemble_weights.csv", index=False)
weights

## 14. Final comparison including ensemble

In [ ]:
strategy_return, market_return, strategy_vol, market_vol, vol_ratio, sharpe, score = metrics(y_val, best_alloc)

ensemble_result = pd.DataFrame([{
    "Model": "Optimized Ensemble",
    "Best parameters": "weighted combination of tuned models",
    "Train RMSE": np.nan,
    "Validation RMSE": rmse(y_val, best_pred),
    "Overfit ratio": np.nan,
    "Strategy return %": strategy_return * 100,
    "Market return %": market_return * 100,
    "Vol ratio %": vol_ratio * 100,
    "Vol constraint": "PASS" if vol_ratio <= 1.20 else "FAIL",
    "Sharpe": sharpe,
    "Model Score": score
}])

final_results = pd.concat([results, ensemble_result], ignore_index=True).sort_values("Model Score", ascending=False)
final_results.to_csv("final_model_comparison.csv", index=False)
final_results

## 15. Final cumulative return including ensemble

This is one of the most important graphs in the project.

- x-axis: validation day
- y-axis: cumulative return
- it compares the market, tuned individual models, and optimized ensemble

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cumulative(y_val.values), label="Market")

for name in allocs:
    plt.plot(cumulative(allocs[name] * y_val.values), label=name)

plt.plot(cumulative(best_alloc * y_val.values), label="Optimized Ensemble")

plt.title("Final cumulative return comparison")
plt.xlabel("validation day")
plt.ylabel("cumulative return")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Final submission

The final ensemble is retrained on the full training set, then used to predict allocations for the test set.

The output is an allocation between 0 and 2.

In [ ]:
test_parts = []

for name, w in zip(model_names, best_w):
    model = clone(best_models[name])
    model.fit(X, y)
    test_parts.append(w * model.predict(X_test))

test_pred = np.sum(test_parts, axis=0)
test_alloc = allocation(test_pred)

submission = pd.DataFrame({
    "date_id": test["date_id"],
    "allocation": test_alloc
})

submission.to_csv("submission.csv", index=False)
submission

# Final conclusion

This compact project shows a logical ML workflow:

1. The data is noisy and volatile, so several robust models were tested.
2. Missing values were handled with a 60% filter, missing indicators, forward fill, and median fill.
3. Feature engineering was added to summarize row behavior and financial groups.
4. Hyperparameters were tuned on the validation period.
5. Models were compared using RMSE, overfitting ratio, return, volatility ratio, Sharpe, and Model Score.
6. The final ensemble was selected because it optimizes risk-adjusted performance under the 120% volatility constraint.
7. The final output is a continuous allocation between 0 and 2.